Sections 1–4: setup, mappings, normalization, context
Sections 5–6: prompt configuration + prompt builder
Sections 7–8: controlled test cases


# ============================================================
# PROMPT MODULE — CELL 1
# Environment & Project Paths
# ============================================================

In [1]:
# ============================================================
# PROMPT MODULE — CELL 23
# Install Google GenAI SDK
# ============================================================

!pip -q install -U google-genai

In [2]:


import os
import json
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project"
)

# Main project directories
DATA_DIR = PROJECT_ROOT / "01_Data"
NOTEBOOK_DIR = PROJECT_ROOT / "02_Notebooks"
CODE_DIR = PROJECT_ROOT / "03_Code"
MODEL_DIR = PROJECT_ROOT / "05_Models"

# Prompt-specific directories
PROMPT_CODE_DIR = CODE_DIR / "Prompt"
PROMPT_NOTEBOOK_DIR = NOTEBOOK_DIR / "Prompt"

# Create Prompt code directory if it does not exist
PROMPT_CODE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Verify project structure
# ------------------------------------------------------------

print("Project root:")
print(PROJECT_ROOT)

print("\nProject root exists:", PROJECT_ROOT.exists())

print("\nMain directories:")
for path in [DATA_DIR, NOTEBOOK_DIR, CODE_DIR, MODEL_DIR]:
    print(f"{path.name:15s} -> {path.exists()}")

print("\nPrompt directories:")
print(f"Prompt code     -> {PROMPT_CODE_DIR}")
print(f"Prompt notebook -> {PROMPT_NOTEBOOK_DIR}")


# ============================================================
# Import reusable Prompt modules
# ============================================================

import sys

CODE_DIR_STR = str(CODE_DIR)

if CODE_DIR_STR not in sys.path:
    sys.path.insert(0, CODE_DIR_STR)


from Prompt.mappings import (
    EEG_GESTURE_MAP,
    EMG_LABEL_MAP,
    SEMANTIC_DISPLAY_NAMES,
)

from Prompt.normalization import (
    normalize_eeg_output,
    normalize_emg_output,
)

from Prompt.context import (
    build_vision_context,
    build_multimodal_context,
)

from Prompt.prompt_builder import (
    build_reasoning_prompt,
)

from Prompt.prompt_config import (
    PROMPT_VERSION,
    SYSTEM_ROLE,
    OUTPUT_INSTRUCTIONS,
)

from Prompt.schema import (
    LLM_OUTPUT_FIELDS,
    VALID_DECISIONS,
    create_empty_reasoning_result,
    validate_reasoning_result,
)

from Prompt.fusion import (
    neural_gesture_agreement,
    vision_has_object,
    vision_has_active_interaction,
    object_matches_instruction,
    deterministic_fusion,
)

from Prompt.llm_reasoner import (
    DEFAULT_MODEL,
    REASONING_SCHEMA,
    create_gemini_client,
    run_reasoning,
)

print("Prompt modules imported successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root:
/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project

Project root exists: True

Main directories:
01_Data         -> True
02_Notebooks    -> True
03_Code         -> True
05_Models       -> True

Prompt directories:
Prompt code     -> /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/03_Code/Prompt
Prompt notebook -> /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/02_Notebooks/Prompt
Prompt modules imported successfully.


In [3]:
# ============================================================
# PROMPT MODULE — CELL 2
# Fixed Semantic Label Mappings
# ============================================================

# ------------------------------------------------------------
# EEG / Dataset gesture mapping
# ------------------------------------------------------------



print("EEG / Dataset Gesture Mapping")
print("-" * 40)

for raw_label, semantic_label in EEG_GESTURE_MAP.items():
    print(f"{raw_label} -> {semantic_label}")


print("\nEMG Decoder Mapping")
print("-" * 40)

for raw_label, semantic_label in EMG_LABEL_MAP.items():
    print(f"{raw_label} -> {semantic_label}")

EEG / Dataset Gesture Mapping
----------------------------------------
G1 -> large_diameter_grasp
G2 -> medium_diameter_grasp
G3 -> three_finger_sphere_grasp
G4 -> prismatic_pinch_grasp
G5 -> power_grasp
G6 -> cut_grasp
G7 -> open_hand

EMG Decoder Mapping
----------------------------------------
0 -> large_diameter_grasp
1 -> medium_diameter_grasp
2 -> three_finger_sphere_grasp
3 -> prismatic_pinch_grasp


# ============================================================
# PROMPT MODULE — CELL 3
# Decoder Output Normalization
# ============================================================

In [4]:


# ------------------------------------------------------------
# Test normalization
# ------------------------------------------------------------

test_eeg = normalize_eeg_output("G3", confidence=0.82)

test_emg = normalize_emg_output(2, confidence=0.91)

print("Normalized EEG:")
print(json.dumps(test_eeg, indent=2))

print("\nNormalized EMG:")
print(json.dumps(test_emg, indent=2))

Normalized EEG:
{
  "raw_label": "G3",
  "semantic_label": "three_finger_sphere_grasp",
  "confidence": 0.82
}

Normalized EMG:
{
  "raw_label": 2,
  "semantic_label": "three_finger_sphere_grasp",
  "confidence": 0.91
}


# ============================================================
# PROMPT MODULE — CELL 4
# Multimodal Context Builder
# ============================================================

In [5]:



# ------------------------------------------------------------
# Controlled example
# ------------------------------------------------------------

eeg_result = normalize_eeg_output(
    "G3",
    confidence=0.82
)

emg_result = normalize_emg_output(
    2,
    confidence=0.91
)

vision_result = build_vision_context(
    handedness="right",
    contact_state=1,
    grasp_type="power_grasp",
    primary_object="coffee cup",
    secondary_object=None,
    hand_object_relation="holding",
)

multimodal_context = build_multimodal_context(
    task_instruction="Pick up the coffee cup.",
    eeg_output=eeg_result,
    emg_output=emg_result,
    vision_output=vision_result,
)

print(json.dumps(multimodal_context, indent=2))

{
  "task": {
    "instruction": "Pick up the coffee cup."
  },
  "eeg": {
    "raw_label": "G3",
    "semantic_label": "three_finger_sphere_grasp",
    "confidence": 0.82
  },
  "emg": {
    "raw_label": 2,
    "semantic_label": "three_finger_sphere_grasp",
    "confidence": 0.91
  },
  "vision": {
    "handedness": "right",
    "contact_state": 1,
    "grasp_type": "power_grasp",
    "primary_object": "coffee cup",
    "secondary_object": null,
    "hand_object_relation": "holding"
  }
}


# ============================================================
# PROMPT MODULE — CELL 5
# Prompt Configuration
# ============================================================


In [6]:

print("Prompt version:", PROMPT_VERSION)
print("\nSystem role:")
print(SYSTEM_ROLE)

Prompt version: v1.0

System role:

You are a multimodal human-intention reasoning assistant.

Your task is to reason over four sources of information:

1. User instruction
2. EEG-based gesture prediction
3. EMG-based gesture prediction
4. Vision-based hand-object interaction information

EEG and EMG predictions are gesture-level predictions.
They must not automatically be interpreted as high-level
intentions such as grasp, release, or neutral.

Vision describes the observed hand-object interaction.

Use the available evidence to determine:
- the likely intended action,
- whether the observations are mutually consistent,
- whether the requested action is supported by the observations,
- and whether the system should proceed, wait, or reject the action.

Do not invent information that is not present.



# ============================================================
# PROMPT MODULE — CELL 6
# Structured Multimodal Prompt Builder
# ============================================================

In [7]:


def build_reasoning_prompt(multimodal_context):
    """
    Convert a standardized multimodal context dictionary
    into a structured reasoning prompt.

    This function does not perform inference.
    It only constructs the prompt.
    """

    if not isinstance(multimodal_context, dict):
        raise TypeError("multimodal_context must be a dictionary.")

    task = multimodal_context.get("task", {})
    eeg = multimodal_context.get("eeg", {})
    emg = multimodal_context.get("emg", {})
    vision = multimodal_context.get("vision", {})

    instruction = task.get("instruction", "")

    # --------------------------------------------------------
    # EEG
    # --------------------------------------------------------

    eeg_raw = eeg.get("raw_label")
    eeg_semantic = eeg.get("semantic_label")
    eeg_confidence = eeg.get("confidence")

    # --------------------------------------------------------
    # EMG
    # --------------------------------------------------------

    emg_raw = emg.get("raw_label")
    emg_semantic = emg.get("semantic_label")
    emg_confidence = emg.get("confidence")

    # --------------------------------------------------------
    # Vision
    # --------------------------------------------------------

    handedness = vision.get("handedness")
    contact_state = vision.get("contact_state")
    grasp_type = vision.get("grasp_type")
    primary_object = vision.get("primary_object")
    secondary_object = vision.get("secondary_object")
    hand_object_relation = vision.get("hand_object_relation")

    # --------------------------------------------------------
    # Build prompt
    # --------------------------------------------------------

    prompt = f"""
{SYSTEM_ROLE}

========================
USER / TASK
========================

Instruction:
{instruction}

========================
EEG OBSERVATION
========================

Raw label:
{eeg_raw}

Semantic gesture:
{eeg_semantic}

Confidence:
{eeg_confidence}

========================
EMG OBSERVATION
========================

Raw label:
{emg_raw}

Semantic gesture:
{emg_semantic}

Confidence:
{emg_confidence}

========================
VISION OBSERVATION
========================

Handedness:
{handedness}

Contact state:
{contact_state}

Grasp type:
{grasp_type}

Primary object:
{primary_object}

Secondary object:
{secondary_object}

Hand-object relationship:
{hand_object_relation}

========================
REASONING TASK
========================

Analyze the multimodal observations.

1. Determine the likely high-level intention/action.
2. Compare the user instruction with the EEG prediction.
3. Compare the user instruction with the EMG prediction.
4. Determine whether the observed hand-object interaction supports
   the requested action.
5. Identify disagreements or missing evidence.
6. Decide whether the system should VALIDATE, WAIT, or REJECT
   the proposed action.

{OUTPUT_INSTRUCTIONS}
""".strip()

    return prompt


# ------------------------------------------------------------
# Generate prompt for our test context
# ------------------------------------------------------------

test_prompt = build_reasoning_prompt(multimodal_context)

print(test_prompt)

You are a multimodal human-intention reasoning assistant.

Your task is to reason over four sources of information:

1. User instruction
2. EEG-based gesture prediction
3. EMG-based gesture prediction
4. Vision-based hand-object interaction information

EEG and EMG predictions are gesture-level predictions.
They must not automatically be interpreted as high-level
intentions such as grasp, release, or neutral.

Vision describes the observed hand-object interaction.

Use the available evidence to determine:
- the likely intended action,
- whether the observations are mutually consistent,
- whether the requested action is supported by the observations,
- and whether the system should proceed, wait, or reject the action.

Do not invent information that is not present.


USER / TASK

Instruction:
Pick up the coffee cup.

EEG OBSERVATION

Raw label:
G3

Semantic gesture:
three_finger_sphere_grasp

Confidence:
0.82

EMG OBSERVATION

Raw label:
2

Semantic gesture:
three_finger_sphere_grasp

C

# ============================================================
# PROMPT MODULE — CELL 7
# Controlled Multimodal Test Cases
# ============================================================

In [8]:


def create_test_context(
    instruction,
    eeg_label,
    eeg_confidence,
    emg_label,
    emg_confidence,
    vision_kwargs,
):
    """
    Convenience function for constructing a complete
    normalized multimodal test case.
    """

    eeg_output = normalize_eeg_output(
        eeg_label,
        confidence=eeg_confidence
    )

    emg_output = normalize_emg_output(
        emg_label,
        confidence=emg_confidence
    )

    vision_output = build_vision_context(
        **vision_kwargs
    )

    return build_multimodal_context(
        task_instruction=instruction,
        eeg_output=eeg_output,
        emg_output=emg_output,
        vision_output=vision_output,
    )


# ------------------------------------------------------------
# CASE 1 — Strong multimodal agreement
# ------------------------------------------------------------

case_1 = create_test_context(
    instruction="Pick up the coffee cup.",
    eeg_label="G3",
    eeg_confidence=0.94,
    emg_label=2,
    emg_confidence=0.96,
    vision_kwargs={
        "handedness": "right",
        "contact_state": 1,
        "grasp_type": "three_finger_sphere_grasp",
        "primary_object": "coffee cup",
        "secondary_object": None,
        "hand_object_relation": "holding",
    },
)


# ------------------------------------------------------------
# CASE 2 — EEG / EMG disagreement
# ------------------------------------------------------------

case_2 = create_test_context(
    instruction="Pick up the coffee cup.",
    eeg_label="G3",
    eeg_confidence=0.91,
    emg_label=0,
    emg_confidence=0.89,
    vision_kwargs={
        "handedness": "right",
        "contact_state": 1,
        "grasp_type": "large_diameter_grasp",
        "primary_object": "coffee cup",
        "secondary_object": None,
        "hand_object_relation": "holding",
    },
)


# ------------------------------------------------------------
# CASE 3 — Vision does not support requested object
# ------------------------------------------------------------

case_3 = create_test_context(
    instruction="Pick up the coffee cup.",
    eeg_label="G3",
    eeg_confidence=0.93,
    emg_label=2,
    emg_confidence=0.94,
    vision_kwargs={
        "handedness": "right",
        "contact_state": 0,
        "grasp_type": None,
        "primary_object": "mobile phone",
        "secondary_object": None,
        "hand_object_relation": "near",
    },
)


# ------------------------------------------------------------
# CASE 4 — Open hand / no active contact
# ------------------------------------------------------------

case_4 = create_test_context(
    instruction="Pick up the coffee cup.",
    eeg_label="G7",
    eeg_confidence=0.95,
    emg_label=0,
    emg_confidence=0.61,
    vision_kwargs={
        "handedness": "right",
        "contact_state": 0,
        "grasp_type": None,
        "primary_object": "coffee cup",
        "secondary_object": None,
        "hand_object_relation": "near",
    },
)


# ------------------------------------------------------------
# CASE 5 — Low-confidence neural predictions
# ------------------------------------------------------------

case_5 = create_test_context(
    instruction="Pick up the coffee cup.",
    eeg_label="G3",
    eeg_confidence=0.52,
    emg_label=2,
    emg_confidence=0.55,
    vision_kwargs={
        "handedness": "right",
        "contact_state": 1,
        "grasp_type": "three_finger_sphere_grasp",
        "primary_object": "coffee cup",
        "secondary_object": None,
        "hand_object_relation": "holding",
    },
)


print("Created test cases:")
print("Case 1:", case_1["task"]["instruction"])
print("Case 2:", case_2["task"]["instruction"])
print("Case 3:", case_3["task"]["instruction"])
print("Case 4:", case_4["task"]["instruction"])
print("Case 5:", case_5["task"]["instruction"])

Created test cases:
Case 1: Pick up the coffee cup.
Case 2: Pick up the coffee cup.
Case 3: Pick up the coffee cup.
Case 4: Pick up the coffee cup.
Case 5: Pick up the coffee cup.


# ============================================================
# PROMPT MODULE — CELL 8
# Inspect Test Cases
# ============================================================

In [9]:


test_cases = {
    "case_1_agreement": case_1,
    "case_2_eeg_emg_disagreement": case_2,
    "case_3_vision_object_mismatch": case_3,
    "case_4_open_hand": case_4,
    "case_5_low_confidence": case_5,
}

for name, context in test_cases.items():

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print(
        json.dumps(
            context,
            indent=2
        )
    )


CASE_1_AGREEMENT
{
  "task": {
    "instruction": "Pick up the coffee cup."
  },
  "eeg": {
    "raw_label": "G3",
    "semantic_label": "three_finger_sphere_grasp",
    "confidence": 0.94
  },
  "emg": {
    "raw_label": 2,
    "semantic_label": "three_finger_sphere_grasp",
    "confidence": 0.96
  },
  "vision": {
    "handedness": "right",
    "contact_state": 1,
    "grasp_type": "three_finger_sphere_grasp",
    "primary_object": "coffee cup",
    "secondary_object": null,
    "hand_object_relation": "holding"
  }
}

CASE_2_EEG_EMG_DISAGREEMENT
{
  "task": {
    "instruction": "Pick up the coffee cup."
  },
  "eeg": {
    "raw_label": "G3",
    "semantic_label": "three_finger_sphere_grasp",
    "confidence": 0.91
  },
  "emg": {
    "raw_label": 0,
    "semantic_label": "large_diameter_grasp",
    "confidence": 0.89
  },
  "vision": {
    "handedness": "right",
    "contact_state": 1,
    "grasp_type": "large_diameter_grasp",
    "primary_object": "coffee cup",
    "secondary_obje

# ============================================================
# PROMPT MODULE — CELL 9
# LLM Output Schema
# ============================================================

In [10]:


print("Required LLM output fields:")
for field in LLM_OUTPUT_FIELDS:
    print(" -", field)

print("\nValid decisions:")
print(VALID_DECISIONS)

Required LLM output fields:
 - inferred_intention
 - requested_action
 - eeg_consistent
 - emg_consistent
 - vision_consistent
 - overall_decision
 - confidence
 - reason

Valid decisions:
{'REJECT', 'WAIT', 'VALID'}


# ============================================================
# PROMPT MODULE — CELL 10
# Test LLM Output Validation
# ============================================================


In [11]:

valid_test_result = {
    "inferred_intention": "grasp",
    "requested_action": "pick up coffee cup",
    "eeg_consistent": True,
    "emg_consistent": True,
    "vision_consistent": True,
    "overall_decision": "VALID",
    "confidence": 0.91,
    "reason": (
        "The instruction, neural predictions, and observed "
        "hand-object interaction are consistent."
    ),
}

try:
    validate_reasoning_result(valid_test_result)
    print("VALID TEST RESULT: PASSED")

except Exception as e:
    print("VALID TEST RESULT: FAILED")
    print("Error:", e)

VALID TEST RESULT: PASSED


# ============================================================
# PROMPT MODULE — CELL 11
# Deterministic Multimodal Fusion
# ============================================================

# ============================================================
# PROMPT MODULE — CELL 12
# Test Deterministic Fusion
# ============================================================

In [12]:


print("=" * 75)
print("DETERMINISTIC FUSION TESTS")
print("=" * 75)

for name, context in test_cases.items():

    result = deterministic_fusion(context)

    print("\n" + "-" * 75)
    print(name)

    print(
        "EEG:",
        context["eeg"]["semantic_label"],
        f"(confidence={context['eeg']['confidence']})"
    )

    print(
        "EMG:",
        context["emg"]["semantic_label"],
        f"(confidence={context['emg']['confidence']})"
    )

    print(
        "Vision object:",
        context["vision"]["primary_object"]
    )

    print(
        "Vision contact:",
        context["vision"]["contact_state"]
    )

    print(
        "Neural agreement:",
        result["neural_gesture_agreement"]
    )

    print(
        "Average neural confidence:",
        round(result["average_neural_confidence"], 3)
    )

    print(
        "Vision object matches instruction:",
        result["vision_object_matches_instruction"]
    )

    print(
        "FINAL BASELINE DECISION:",
        result["decision"]
    )

DETERMINISTIC FUSION TESTS

---------------------------------------------------------------------------
case_1_agreement
EEG: three_finger_sphere_grasp (confidence=0.94)
EMG: three_finger_sphere_grasp (confidence=0.96)
Vision object: coffee cup
Vision contact: 1
Neural agreement: True
Average neural confidence: 0.95
Vision object matches instruction: True
FINAL BASELINE DECISION: VALID

---------------------------------------------------------------------------
case_2_eeg_emg_disagreement
EEG: three_finger_sphere_grasp (confidence=0.91)
EMG: large_diameter_grasp (confidence=0.89)
Vision object: coffee cup
Vision contact: 1
Neural agreement: False
Average neural confidence: 0.9
Vision object matches instruction: True
FINAL BASELINE DECISION: WAIT

---------------------------------------------------------------------------
case_3_vision_object_mismatch
EEG: three_finger_sphere_grasp (confidence=0.93)
EMG: three_finger_sphere_grasp (confidence=0.94)
Vision object: mobile phone
Vision cont

# ============================================================
# PROMPT MODULE — CELL 13
# Generate Prompts for All Test Cases
# ============================================================

In [13]:


generated_prompts = {}

for name, context in test_cases.items():

    generated_prompts[name] = build_reasoning_prompt(
        context
    )

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    print(generated_prompts[name])


CASE_1_AGREEMENT
You are a multimodal human-intention reasoning assistant.

Your task is to reason over four sources of information:

1. User instruction
2. EEG-based gesture prediction
3. EMG-based gesture prediction
4. Vision-based hand-object interaction information

EEG and EMG predictions are gesture-level predictions.
They must not automatically be interpreted as high-level
intentions such as grasp, release, or neutral.

Vision describes the observed hand-object interaction.

Use the available evidence to determine:
- the likely intended action,
- whether the observations are mutually consistent,
- whether the requested action is supported by the observations,
- and whether the system should proceed, wait, or reject the action.

Do not invent information that is not present.


USER / TASK

Instruction:
Pick up the coffee cup.

EEG OBSERVATION

Raw label:
G3

Semantic gesture:
three_finger_sphere_grasp

Confidence:
0.94

EMG OBSERVATION

Raw label:
2

Semantic gesture:
three_fing

In [ ]:
# ============================================================
# PROMPT MODULE — CELL 24
# Gemini API configuration
# ============================================================

from google import genai
from google.colab import userdata


GEMINI_API_KEY = "GIVE KEY HERE"

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY was not found in Colab Secrets."
    )


gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


# ============================================================
# PROMPT MODULE — CELL 27
# Initialize reasoning model
# ============================================================

In [15]:


gemini_reasoner = create_gemini_client(
    GEMINI_API_KEY
)

print("Reasoning model initialized.")
print("Model:", DEFAULT_MODEL)

Reasoning model initialized.
Model: gemini-3.1-flash-lite


# ============================================================
# PROMPT MODULE — CELL 28
# First real LLM reasoning test
# ============================================================

In [16]:


test_context = test_cases[
    "case_1_agreement"
]

test_prompt = build_reasoning_prompt(
    test_context
)

response = gemini_reasoner.models.generate_content(
    model=DEFAULT_MODEL,
    contents=test_prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": REASONING_SCHEMA,
    },
)

print("Raw model response:")
print(response.text)

Raw model response:
{
  "inferred_intention": "pick_up_coffee_cup",
  "requested_action": "pick_up_coffee_cup",
  "eeg_consistent": true,
  "emg_consistent": true,
  "vision_consistent": false,
  "overall_decision": "REJECT",
  "confidence": 0.95,
  "reason": "The user requested to pick up the coffee cup, but the vision observation indicates the hand is already holding the cup, making the action redundant or already completed."
}


# ============================================================
# PROMPT MODULE — CELL 29
# Parse and validate LLM output
# ============================================================


In [17]:

import json


raw_result = response.text

reasoning_result = json.loads(
    raw_result
)

validate_reasoning_result(
    reasoning_result
)

print("LLM output validation: PASSED")

print("\nStructured reasoning result:")
print(
    json.dumps(
        reasoning_result,
        indent=2
    )
)

LLM output validation: PASSED

Structured reasoning result:
{
  "inferred_intention": "pick_up_coffee_cup",
  "requested_action": "pick_up_coffee_cup",
  "eeg_consistent": true,
  "emg_consistent": true,
  "vision_consistent": false,
  "overall_decision": "REJECT",
  "confidence": 0.95,
  "reason": "The user requested to pick up the coffee cup, but the vision observation indicates the hand is already holding the cup, making the action redundant or already completed."
}


# ============================================================
# PROMPT MODULE — CELL 30
# Compare LLM and deterministic decisions
# ============================================================

In [18]:


baseline_result = deterministic_fusion(
    test_context
)

print("=" * 70)
print("MULTIMODAL REASONING COMPARISON")
print("=" * 70)

print(
    "\nDeterministic baseline:",
    baseline_result["decision"]
)

print(
    "LLM decision:",
    reasoning_result["overall_decision"]
)

print(
    "\nLLM inferred intention:",
    reasoning_result["inferred_intention"]
)

print(
    "LLM confidence:",
    reasoning_result["confidence"]
)

print(
    "\nLLM reason:"
)

print(
    reasoning_result["reason"]
)

MULTIMODAL REASONING COMPARISON

Deterministic baseline: VALID
LLM decision: REJECT

LLM inferred intention: pick_up_coffee_cup
LLM confidence: 0.95

LLM reason:
The user requested to pick up the coffee cup, but the vision observation indicates the hand is already holding the cup, making the action redundant or already completed.


ENDING CONTEXT


PROJECT CONTEXT — PROMPT MODULE WORK DONE TODAY

Project: Multimodal EEG + EMG + Image + Text research prototype.

Architecture currently agreed:
EEG → EEG decoder → gesture + confidence ┐
EMG → EMG decoder → gesture + confidence ├→ multimodal context → LLM → structured reasoning → deterministic fusion → final decision
Image → Vision model → semantic vision context ┘
Text instruction ───────────────────────────┘

PROMPT MODULE:
Created:
03_Code/Prompt/
├── __init__.py
├── mappings.py
├── normalization.py
├── context.py
├── prompt_config.py
├── prompt_builder.py
├── schema.py
├── fusion.py
└── llm_reasoner.py

Canonical EEG mapping:
G1 = large_diameter_grasp
G2 = medium_diameter_grasp
G3 = three_finger_sphere_grasp
G4 = prismatic_pinch_grasp
G5 = power_grasp
G6 = cut_grasp
G7 = open_hand

EMG mapping:
0 = large_diameter_grasp
1 = medium_diameter_grasp
2 = three_finger_sphere_grasp
3 = prismatic_pinch_grasp

Normalization functions:
normalize_eeg_output(raw_label, confidence)
normalize_emg_output(raw_label, confidence)

Vision context schema:
handedness
contact_state
grasp_type
primary_object
secondary_object
hand_object_relation

Multimodal context contains:
task/instruction
eeg output
emg output
vision output

Prompt builder:
build_reasoning_prompt(context)
creates a structured prompt with task + EEG + EMG + vision information.

Important semantic rule:
EEG/EMG gesture classes MUST NOT automatically be interpreted as high-level
grasp/release/neutral intentions. They are gesture-level predictions.
Vision describes observed hand/object interaction.
The LLM is the reasoning layer, NOT a raw EEG/EMG classifier.

LLM output schema:
inferred_intention
requested_action
eeg_consistent
emg_consistent
vision_consistent
overall_decision
confidence
reason

overall_decision ∈ {VALID, WAIT, REJECT}
confidence ∈ [0,1]

Created deterministic fusion baseline:
- checks EEG/EMG semantic agreement
- checks vision object presence
- checks active interaction/contact
- checks object matches instruction
- checks average neural confidence
- returns VALID when conditions are satisfied, otherwise WAIT.

Test cases:
1. Strong multimodal agreement → VALID
2. EEG/EMG disagreement → WAIT
3. Vision object mismatch → WAIT
4. Open hand/no active contact → WAIT
5. Low-confidence neural predictions → WAIT

LLM integration:
Using google-genai / Gemini.
Current model variable:
DEFAULT_MODEL = "gemini-3.1-flash-lite"

Working API call:
client.models.generate_content(
    model=DEFAULT_MODEL,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": REASONING_SCHEMA,
    },
)

The earlier config using:
config={"response_format": {"text": {...}}}
did NOT work and should NOT be reused.

LLM response is parsed with json.loads() and checked using
validate_reasoning_result().

First real LLM test worked, but LLM returned REJECT while deterministic
fusion returned VALID because the test vision state said the hand was already
"holding" the cup. We concluded this is a semantic/temporal interpretation
issue, NOT an API failure:
a single image showing current contact/holding does NOT prove that the
requested pickup action has already been completed.

Important future prompt clarification:
contact_state/relation describe the CURRENT visual state.
Do not infer temporal completion of an action solely from "holding" or contact.

CURRENT PLAN:
We decided to implement the IMAGE branch next, before integrating all three
modalities into the LLM.

Image branch goal:
Image → Vision model → standardized semantic vision context.

Initial vision output should look like:
{
    "handedness": "...",
    "contact_state": "...",
    "grasp_type": "...",
    "primary_object": "...",
    "secondary_object": "...",
    "hand_object_relation": "..."
}

We will initially test one image at a time rather than loading the entire
Hands23 dataset. After the image pipeline works, test it on selected Hands23
images.

Likely Vision code structure:
03_Code/Vision/
├── __init__.py
├── vision_config.py
├── vision_model.py
├── vision_parser.py
└── vision_inference.py

After Image works, connect:
EEG decoder output + EMG decoder output + Image/Vision semantics + Text
→ Prompt Builder → Gemini LLM → structured reasoning → deterministic fusion.